# Volume 2. LRI Parameter Lab

Question: how do refractory period and inhibition strength change cued sequence recall?

This lab turns LRI from a single switch into a small parameter surface. Each row trains the same short sequence, enables LRI, runs a traced recall, and records how many positions match in order.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from neural_assemblies.assembly_calculus import overlap, sequence_memorize
from neural_assemblies.assembly_calculus.tracing import RecallSweepConfig, lri_recall_sweep, ordered_recall_trace
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import animate_assembly_trace, plot_parameter_heatmap, plot_recall_trace, plot_winner_turnover


In [ ]:
REFRACTORY_PERIODS = [1, 2, 3]
INHIBITION_STRENGTHS = [20.0, 100.0, 300.0]
SEQUENCE = ("morning", "coffee", "code", "notes")

configs = [
    RecallSweepConfig(
        refractory_period=period,
        inhibition_strength=strength,
        beta_boost=0.25,
        n=3_500,
        k=45,
        rounds_per_step=7,
        repetitions=2,
        sequence=SEQUENCE,
        seed=19,
    )
    for period in REFRACTORY_PERIODS
    for strength in INHIBITION_STRENGTHS
]
results = pd.DataFrame(lri_recall_sweep(configs))
results


In [ ]:
pivot = results.pivot(index="refractory_period", columns="inhibition_strength", values="ordered_matches")
ax, values = plot_parameter_heatmap(
    pivot.values,
    x_labels=pivot.columns,
    y_labels=pivot.index,
    title="Ordered recall matches",
    value_format=".0f",
    cbar_label="matched positions",
)
ax.set_xlabel("inhibition strength")
ax.set_ylabel("refractory period")
plt.show()
plt.close(ax.figure)


Choose one setting and inspect the actual trace. A good parameter row is still just a summary until the recalled states are compared with stored items.


In [ ]:
N = 3_500
K = 45
brain = Brain(p=0.05, save_winners=True, seed=19, engine="numpy_sparse")
for stimulus in SEQUENCE:
    brain.add_stimulus(stimulus, K)
brain.add_area("SEQ", N, K, 0.10)
memorized = sequence_memorize(
    brain,
    list(SEQUENCE),
    "SEQ",
    rounds_per_step=7,
    repetitions=2,
    phase_b_ratio=0.5,
    beta_boost=0.25,
)
brain.set_lri("SEQ", refractory_period=2, inhibition_strength=100.0)
recall_trace = ordered_recall_trace(
    brain,
    "SEQ",
    SEQUENCE[0],
    max_steps=len(SEQUENCE) + 4,
    known_assemblies=list(memorized),
)

pd.DataFrame(recall_trace.to_records())


In [ ]:
fig, animation = animate_assembly_trace(
    recall_trace,
    N,
    title="Detailed LRI recall trace",
    color="#7768ae",
    interval=850,
)
html = HTML(animation.to_jshtml())
plt.close(fig)
html


In [ ]:
ax, matrix = plot_recall_trace(
    recall_trace.assemblies,
    list(memorized),
    recalled_labels=[f"step {step.round_index}" for step in recall_trace],
    known_labels=SEQUENCE,
)
ax.set_title("Detailed recall-to-memory overlap")
plt.show()
plt.close(ax.figure)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
plot_winner_turnover(recall_trace, ax=axes[0], title="LRI winner turnover")
axes[1].plot(
    [step.round_index for step in recall_trace],
    [max(overlap(step.assembly, known) for known in memorized) for step in recall_trace],
    marker="o",
    color="#7768ae",
)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("recall step")
axes[1].set_ylabel("best overlap")
axes[1].set_title("Distance from stored items")
axes[1].grid(axis="y", alpha=0.25)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
plt.show()
plt.close(fig)


The useful lesson is qualitative: too little inhibition can leave the current attractor sticky, while too much can push activity into drift. The heatmap tells you where to inspect; the trace tells you what happened.
